In [14]:
pip install thop torchinfo

Note: you may need to restart the kernel to use updated packages.


In [15]:
import torch
import time
import os
from thop import profile

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_model_stats(model, input_size=(1, 3, 224, 224), repetitions=100, warmup=10):
    model = model.to(device)
    model.eval()

    dummy_input = torch.randn(input_size).to(device)

    # -----------------------
    # 1. Parameters
    # -----------------------
    params = sum(p.numel() for p in model.parameters()) / 1e6

    # -----------------------
    # 2. FLOPs (safe)
    # -----------------------
    try:
        flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
        flops = flops / 1e9
    except Exception as e:
        print("FLOPs Error:", e)
        flops = 0.0

    # -----------------------
    # 3. Model Size
    # -----------------------
    torch.save(model.state_dict(), "temp.pth")
    size_mb = os.path.getsize("temp.pth") / (1024 * 1024)
    os.remove("temp.pth")

    # -----------------------
    # 4. Inference Time (FIXED)
    # -----------------------
    timings = []

    with torch.no_grad():
        # Warm-up
        for _ in range(warmup):
            _ = model(dummy_input)

        if device.type == "cuda":
            starter, ender = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)

            for _ in range(repetitions):
                starter.record()
                _ = model(dummy_input)
                ender.record()
                torch.cuda.synchronize()
                timings.append(starter.elapsed_time(ender))
        else:
            for _ in range(repetitions):
                start = time.time()
                _ = model(dummy_input)
                end = time.time()
                timings.append((end - start) * 1000)

    inf_time = sum(timings) / len(timings)

    # -----------------------
    # 5. GPU Memory
    # -----------------------
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = model(dummy_input)
        mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        mem = 0.0

    return {
        "Params(M)": round(params, 2),
        "FLOPs(G)": round(flops, 2),
        "Size(MB)": round(size_mb, 2),
        "Inf Time(ms)": round(inf_time, 2),
        "GPU Mem(GB)": round(mem, 2)
    }

In [16]:
import torch
import torch.nn as nn
import torchvision.models as models

class ResNet_ViT(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        # ----------------------------
        # ResNet50 Backbone
        # ----------------------------
        resnet = models.resnet50(weights=None)
        self.res_features = nn.Sequential(*list(resnet.children())[:-2])
        self.res_pool = nn.AdaptiveAvgPool2d(1)
        self.res_out = 2048

        # ----------------------------
        # Vision Transformer
        # ----------------------------
        vit = models.vit_b_16(weights=None)
        self.vit = vit
        self.vit_out = vit.heads.head.in_features  # 768
        self.vit.heads = nn.Identity()  # remove classifier

        # ----------------------------
        # Fusion Weights
        # ----------------------------
        self.weights = nn.Parameter(torch.ones(2))

        # ----------------------------
        # Classifier
        # ----------------------------
        self.classifier = nn.Sequential(
            nn.Linear(self.res_out + self.vit_out, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # ResNet branch
        r = self.res_features(x)
        r = self.res_pool(r)
        r = torch.flatten(r, 1)

        # ViT branch
        v = self.vit(x)

        # Normalize weights
        w = torch.softmax(self.weights, dim=0)
        r = w[0] * r
        v = w[1] * v

        # Fusion
        fused = torch.cat([r, v], dim=1)
        return self.classifier(fused)

In [17]:
model = ResNet_ViT(num_classes=2)

stats = compute_model_stats(model, (1, 3, 224, 224))

print(stats)

{'Params(M)': 110.75, 'FLOPs(G)': 15.42, 'Size(MB)': 422.87, 'Inf Time(ms)': 20.85, 'GPU Mem(GB)': 0.69}
